# World session 3 · Links that break

**Goal:** check every link between tables. Course 1 had one table, so nothing could point at nothing. Here 6 million participants point at a fight and a person, and some of those links go nowhere, or go to a duplicate.

**How to use this notebook**
- Click a gray code cell and press **Shift + Enter** to run it. Go top to bottom.
- 🧪 **Your turn** cells contain a `___` blank. Replace it with your answer and run the cell; it tells you whether you're right. Answers are at the bottom if you're stuck.
- Your answers are checked against the world's **answer keys**: the generator's own record of every mess it made. A result you hand in (rows you found) is graded by **precision** (how many of yours are right) and **recall** (how many of the right ones you found).
- Nothing here can change the lake. You read the messy files as a source system delivered them.

> ⚠️ **Mock data only.** Every person, place and record in this world is invented.

In [ ]:
# ▶ SETUP: run this cell first (Shift + Enter). It finds the course files and opens the world's messy files.
SESSION = "3"

import sys
from pathlib import Path

for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / "lakehouse" / "sessions" / "check.py").exists():
        sys.path.insert(0, str(folder))
        break

import re
import pandas as pd
from lakehouse.sessions import check
from lakehouse.sessions.check import ___

con = check.world()       # DuckDB: one view messy_<table> per messy CSV folder, every column as text
print(f"World session {SESSION} ready. Messy tables:", ", ".join(r[0] for r in con.sql("SHOW TABLES").fetchall()))

## 1. A participant links a fight and a person

In [ ]:
con.sql("SELECT participant_id, incident_id, person_id, side, role, status FROM messy_participant USING SAMPLE 5 ROWS (reservoir, 3)").df()

## 2. Links to nothing: orphans
An **anti-join** keeps the rows with no match on the other side. In SQL: `WHERE x NOT IN (SELECT ...)`, or `LEFT JOIN ... WHERE other.id IS NULL`, or DuckDB's `ANTI JOIN`.

🧪 **Your turn #1.** How many participants point at a fight (`incident_id`) that isn't in `messy_incident`?

In [ ]:
answer_1 = ___      # an anti-join of messy_participant to messy_incident on incident_id
check.w3_ex1(answer_1)

🧪 **Your turn #2.** And how many point at a person (`person_id`) who isn't in `messy_person`?

In [ ]:
answer_2 = ___      # the same, on person_id
check.w3_ex2(answer_2)

Don't delete them. A participant whose fight is missing still happened; the fight row was lost upstream. Flag the link and ask the source system.

## 3. Links to the wrong row: duplicates
Some participants point at a person who *is* there, but is a duplicate entered later. The link isn't broken; it's split: one person, two ids. World session 1 found the duplicates; here is that result again.

In [ ]:
con.execute("""
CREATE OR REPLACE MACRO clean_text(s) AS
    CASE WHEN upper(trim(s, ' 	')) IN ('', 'N/A', 'UNKNOWN', 'UNK', '?', 'NULL', 'NONE') THEN NULL ELSE upper(trim(s, ' 	')) END
""")
con.execute("""
CREATE OR REPLACE MACRO read_date(s) AS CASE
    WHEN regexp_matches(trim(s), '^\\d{1,2}/\\d{1,2}/\\d{2}$') THEN          -- 3/22/01: a two-digit year
        make_date(CASE WHEN CAST(split_part(trim(s), '/', 3) AS INT) <= 26 THEN 2000 ELSE 1900 END
                      + CAST(split_part(trim(s), '/', 3) AS INT),
                  CAST(split_part(trim(s), '/', 1) AS INT), CAST(split_part(trim(s), '/', 2) AS INT))
    ELSE COALESCE(try_strptime(trim(s), '%Y-%m-%d'),          -- 2001-03-22
                  try_strptime(trim(s), '%Y-%m-%d %H:%M:%S'), -- 2001-03-22 00:00:00
                  try_strptime(trim(s), '%m/%d/%Y'),          -- 03/22/2001
                  try_strptime(trim(s), '%m-%d-%Y'),          -- 03-22-2001
                  try_strptime(trim(s), '%Y%m%d'),            -- 20010322
                  try_strptime(trim(s), '%d-%b-%Y'),          -- 22-MAR-2001
                  try_strptime(trim(s), '%b %d %Y'),          -- Mar 22 2001
                  try_strptime(trim(s), '%B %d, %Y'))::DATE   -- March 22, 2001
END
""")
duplicates = con.sql("""
    WITH people AS (
        SELECT CAST(person_id AS BIGINT) AS person_id,
               COALESCE('SID ' || nullif(trim(sid_number), ''),
                        clean_text(last_name) || '|' || clean_text(first_name) || '|' || read_date(birth_date)) AS match_key
        FROM messy_person
    )
    SELECT person_id AS dup_id, min(person_id) OVER (PARTITION BY match_key) AS original_id
    FROM people
    WHERE match_key IS NOT NULL
    QUALIFY dup_id <> original_id
""").df()
print(len(duplicates), "duplicates")

🧪 **Your turn #3.** Build `repointed`: for every participant whose `person_id` is a duplicate, its `participant_id` and the `person_id` it should point at (the original). The bar is 99% precision and 85% recall: you can only re-point to duplicates you found.

In [ ]:
repointed = ___      # join messy_participant to duplicates on person_id = dup_id
check.w3_ex3(repointed)

## 4. How the lake does it
Silver flags every orphan (`orphan` in `world_silver.exceptions`) and keeps the row. Gold carries `person_key`, the resolved person, so a fight counts each person once, whichever id the clerk typed.

In [ ]:
from lakehouse import lake
silver = lake.lake_ro_connect()
silver.sql("SELECT table_name, column_name, count(*) AS orphans FROM lake.world_silver.exceptions WHERE rule = 'orphan' GROUP BY ALL ORDER BY orphans DESC LIMIT 6").df()

## Wrap-up
- Check every link with an anti-join; count orphans per link.
- Keep orphans, flagged: the event happened even if its parent row was lost.
- A link to a duplicate isn't broken but split; resolve people first (session 1), then re-point.
- Report what you couldn't re-point, by id.

## Answers
Try each exercise first. Each answer replaces the `___` line in its 🧪 cell.

**#1** Participants whose `incident_id` matches no incident.

```python
answer_1 = con.sql("""
    SELECT count(*) FROM messy_participant p
    WHERE p.incident_id NOT IN (SELECT incident_id FROM messy_incident WHERE incident_id IS NOT NULL)
""").fetchone()[0]
```

**#2** The same on `person_id`.

```python
answer_2 = con.sql("""
    SELECT count(*) FROM messy_participant p
    WHERE p.person_id NOT IN (SELECT person_id FROM messy_person WHERE person_id IS NOT NULL)
""").fetchone()[0]
```

**#3** Join to the duplicates and take the original's id.

```python
repointed = con.sql("""
    SELECT CAST(p.participant_id AS BIGINT) AS participant_id, d.original_id AS person_id
    FROM messy_participant p JOIN duplicates d ON CAST(p.person_id AS BIGINT) = d.dup_id
""").df()
```